# Fikra Phase 1 — Full Benchmark (Build + Run), Colab-only

This single notebook builds the frozen 500-task benchmark from pinned
HuggingFace dataset revisions AND runs all 2,000 Fikra API evaluations,
entirely inside this Colab session. Nothing needs to move between
machines, and none of it touches your own network/data bundle.

**Before running:** add two Colab secrets (key icon, left sidebar):
- `FIKRA_API_KEY` — your Fikra API key
- `HF_TOKEN` — a HuggingFace token with **gated-repo read access**
  enabled (Settings → Access Tokens; needed for `GPQA-Diamond`). Also
  make sure your HF account has accepted GPQA's terms at
  huggingface.co/datasets/Idavidrein/gpqa.

Run all cells top to bottom. The run is resumable: if the Colab
runtime disconnects partway through, just re-run this notebook from
the top — the benchmark build is skipped once it exists (hash-checked),
and the evaluation loop skips any (task_id, model) pair that already
succeeded, so you won't lose progress or re-spend API calls.

The final cell zips everything (frozen benchmark + all 2,000 results +
CSV summary) and downloads it to your machine.


In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================
!pip install -q "datasets>=3.3,<4" "huggingface_hub>=0.28,<1" pandas requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which

In [ ]:
# ============================================================
# 1. SECRETS + LOCAL WORKSPACE (Colab)
# ============================================================
# This notebook builds the frozen benchmark AND runs all 2,000 Fikra
# evaluations in one place, so no dataset/results files need to move
# between machines. Add two Colab secrets (key icon, left sidebar):
#   FIKRA_API_KEY   - your Fikra API key
#   HF_TOKEN        - HuggingFace token with gated-repo read access
#                      (needed for GPQA-Diamond)

import os
import sys
import json
import csv
import base64
import gc
import hashlib
import math
import random
import subprocess
import time
import tarfile
import tempfile
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone, timedelta

import requests


def get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

FIKRA_API_KEY = get_secret("FIKRA_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

if not FIKRA_API_KEY:
    raise RuntimeError("Missing FIKRA_API_KEY. Add it as a Colab secret (key icon, left sidebar).")

if not HF_TOKEN:
    raise RuntimeError("Missing HF_TOKEN. Add it as a Colab secret (key icon, left sidebar).")

FIKRA_BASE_URL = "https://api.fikraapi.co.ke/v1"
HEADERS = {
    "Authorization": f"Bearer {FIKRA_API_KEY}",
    "Content-Type": "application/json",
}

ROOT = Path("/content/fikra_benchmark_phase1")
ROOT.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Workspace:", ROOT)
print("Secrets: loaded")




Python: 3.13.15
Workspace: /content/fikra_benchmark_phase1
Secrets: loaded


In [ ]:
# ============================================================
# 2. FINAL EXPERIMENT CONFIGURATION
# ============================================================

SEED = 20260926
EXPERIMENT_ID = "fikra-phase1-2026-09-26-v1"
RUN_ID = "fikra-benchmark-phase1-2026-09-26-4model"

COUNTS = {
    "MMLU-Pro": 100,
    "GPQA-Diamond": 50,
    "GSM8K": 75,
    "IFEval": 75,
    "LiveCodeBench": 100,
    "BFCL": 50,
    "FRES": 50,
}

FINAL_MODELS = [
    "fikra-flash",
    "fikra-pro-20b",
    "fikra-pro-120b",
    "fikra-qwen-max",
]

MODEL_UPSTREAM = {
    "fikra-flash": "GPT-OSS 20B",
    "fikra-pro-20b": "Qwen3.8 27B",
    "fikra-pro-120b": "GPT-OSS 120B",
    "fikra-qwen-max": "Qwen 3.8 Max",
}

EXPECTED_TASKS = 500
EXPECTED_EVALUATIONS = 2000

PINNED_REVISIONS = {
    "MMLU-Pro": "b189ec765aa7ed75c8acfea42df31fdae71f97be",
    "GPQA-Diamond": "83022cefff930aea54f654c0b282e74b9eeda5c6",
    "GSM8K": "740312add88f781978c0658806c59bc2815b9866",
    "IFEval": "966cd89545d6b6acfd7638bc708b98261ca58e84",
    "LiveCodeBench": "d44be6b144381afa49392b1f0eb424a64a4d8a10",
}

BFCL_REPO_URL = "https://github.com/EnlightenedAI/BFCL.git"
BFCL_COMMIT = "f7cf735"
LCB_REPO = "livecodebench/code_generation_lite"
LCB_RELEASE = "release_v6"

BENCHMARK_JSONL = ROOT / "benchmark_500.jsonl"
BENCHMARK_CSV = ROOT / "benchmark_500.csv"
PUBLIC_MANIFEST_CSV = ROOT / "benchmark_500_public_manifest.csv"
FROZEN_MANIFEST = ROOT / "frozen_manifest.json"
MODEL_REGISTRY_SNAPSHOT = ROOT / "model_registry_snapshot.json"
PRODUCTION_PATH = ROOT / f"{RUN_ID}_results.jsonl"
RUN_MANIFEST_PATH = ROOT / f"{RUN_ID}_manifest.json"
PROGRESS_PATH = ROOT / f"{RUN_ID}_progress.json"
FINAL_CSV = ROOT / f"{RUN_ID}_results.csv"

MAX_OUTPUT_TOKENS = 4096
DEFAULT_REQUEST_INTERVAL = 2.1
RETRYABLE_STATUS_CODES = {408, 429, 500, 502, 503, 504}

assert sum(COUNTS.values()) == 500
assert len(FINAL_MODELS) == 4
assert len(FINAL_MODELS) * 500 == 2000

print("Expected evaluations:", EXPECTED_EVALUATIONS)
print("Result file exists:", PRODUCTION_PATH.exists())




Expected evaluations: 2000
Result file exists: False


In [ ]:
# ============================================================
# 3. REPRODUCIBILITY UTILITIES
# ============================================================

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def canonical_json(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def stable_seed(*parts) -> int:
    value = "::".join(map(str, parts))
    return int(hashlib.sha256(value.encode()).hexdigest()[:12], 16)


def stable_sample(rows, n, seed):
    rows = list(rows)
    if len(rows) < n:
        raise ValueError(f"Need {n} rows; found {len(rows)}")
    rng = random.Random(seed)
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    return [rows[i] for i in idx[:n]]


def stratified_sample(rows, n, key_fn, seed):
    rows = list(rows)
    if len(rows) < n:
        raise ValueError(f"Need {n} rows; found {len(rows)}")
    groups = defaultdict(list)
    for row in rows:
        groups[key_fn(row)].append(row)
    keys = sorted(groups, key=lambda x: str(x))
    rng = random.Random(seed)
    for key in keys:
        rng.shuffle(groups[key])
    total = len(rows)
    quotas = {k: n * len(groups[k]) / total for k in keys}
    take = {k: math.floor(quotas[k]) for k in keys}
    remainder = n - sum(take.values())
    ranked = sorted(keys, key=lambda k: (quotas[k] - take[k], str(k)), reverse=True)
    for k in ranked[:remainder]:
        take[k] += 1
    selected = []
    for k in keys:
        selected.extend(groups[k][:take[k]])
    rng.shuffle(selected)
    assert len(selected) == n
    return selected


def write_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256_file_streaming(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("Utilities ready.")




Utilities ready.


In [ ]:
# ============================================================
# 4. BUILD OR RECOVER THE FROZEN 500-TASK BENCHMARK
# ============================================================

# Pandas/HF are only used if the frozen benchmark is missing.
try:
    import pandas as pd
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
    import pandas as pd

try:
    from datasets import load_dataset
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets>=3.3,<4", "huggingface_hub>=0.28,<1"], check=True)
    from datasets import load_dataset
    from huggingface_hub import HfApi, hf_hub_download


# If the exact frozen benchmark already exists on disk,
# load it and verify its hash. Otherwise assemble it once.

if (
    BENCHMARK_JSONL.exists()
    and FROZEN_MANIFEST.exists()
):
    frozen_meta = read_json(
        FROZEN_MANIFEST
    )

    benchmark_bytes = (
        BENCHMARK_JSONL.read_bytes()
    )

    actual_sha = sha256_bytes(
        benchmark_bytes
    )

    if actual_sha != frozen_meta["benchmark_sha256"]:
        raise RuntimeError(
            "Frozen benchmark hash mismatch. "
            "Do not overwrite it."
        )

    benchmark_df = pd.read_json(
        BENCHMARK_JSONL,
        lines=True,
    )

    assert len(benchmark_df) == 500
    assert benchmark_df["task_id"].is_unique

    print(
        "Recovered frozen benchmark from local disk."
    )
    print(
        "SHA-256:",
        actual_sha,
    )

else:
    print(
        "No frozen benchmark found. "
        "Building it once from pinned sources."
    )

    # -------------------------
    # HF helpers
    # -------------------------

    from huggingface_hub import (
        HfApi,
        hf_hub_download,
    )
    from datasets import load_dataset

    HF_API = HfApi(
        token=HF_TOKEN
    )

    def load_hf(
        repo_id,
        split,
        revision,
        config=None,
        **kwargs,
    ):
        arguments = {
            "split": split,
            "revision": revision,
            "token": HF_TOKEN,
        }

        if config is not None:
            arguments["name"] = config

        arguments.update(kwargs)

        return load_dataset(
            repo_id,
            **arguments,
        )

    # -------------------------
    # MMLU-Pro
    # -------------------------

    mmlu_ds = load_hf(
        "TIGER-Lab/MMLU-Pro",
        "test",
        PINNED_REVISIONS["MMLU-Pro"],
    )

    assert len(mmlu_ds) == 12032

    mmlu_rows = [
        dict(x)
        for x in mmlu_ds
    ]

    mmlu_selected = stratified_sample(
        mmlu_rows,
        100,
        key_fn=lambda x:
            x["category"],
        seed=stable_seed(
            SEED,
            "mmlu_pro",
        ),
    )

    mmlu_tasks = []

    for x in mmlu_selected:

        options = list(
            x["options"]
        )

        labels = [
            chr(
                ord("A") + i
            )
            for i in range(
                len(options)
            )
        ]

        answer_index = int(
            x["answer_index"]
        )

        option_text = "\n".join(
            f"{label}. {option}"
            for label, option
            in zip(
                labels,
                options,
            )
        )

        mmlu_tasks.append({
            "task_id":
                f"MMLUP-{int(x['question_id']):05d}",

            "dataset":
                "MMLU-Pro",

            "dataset_revision":
                PINNED_REVISIONS["MMLU-Pro"],

            "category":
                x["category"],

            "source_id":
                str(x["question_id"]),

            "prompt":
                (
                    f"{x['question']}\n\n"
                    f"{option_text}\n\n"
                    "Answer with the single correct option letter."
                ),

            "reference_answer":
                labels[answer_index],

            "scoring":
                "exact_choice",

            "metadata_json":
                canonical_json({
                    "question_id":
                        x["question_id"],
                    "category":
                        x["category"],
                    "source":
                        x.get("src"),
                    "answer_index":
                        answer_index,
                    "options":
                        options,
                }),
        })

    # -------------------------
    # GPQA Diamond
    # -------------------------

    gpqa_path = hf_hub_download(
        repo_id="Idavidrein/gpqa",
        filename="gpqa_diamond.csv",
        repo_type="dataset",
        revision=PINNED_REVISIONS[
            "GPQA-Diamond"
        ],
        token=HF_TOKEN,
    )

    gpqa_df = pd.read_csv(
        gpqa_path
    )

    # Preserve compatibility with the exact schema
    # captured during the working run.
    question_col = "Question"
    correct_col = "Correct Answer"
    incorrect_cols = [
        "Incorrect Answer 1",
        "Incorrect Answer 2",
        "Incorrect Answer 3",
    ]

    if not all(
        col in gpqa_df.columns
        for col in [
            question_col,
            correct_col,
            *incorrect_cols,
        ]
    ):
        raise RuntimeError(
            "GPQA schema differs from the frozen run's schema: "
            + str(
                gpqa_df.columns.tolist()
            )
        )

    assert len(gpqa_df) == 198

    gpqa_records = gpqa_df.to_dict(
        "records"
    )

    gpqa_group_col = (
        "Subdomain"
        if "Subdomain"
        in gpqa_df.columns
        else None
    )

    if gpqa_group_col:
        gpqa_selected = stratified_sample(
            gpqa_records,
            50,
            key_fn=lambda x:
                x.get(
                    gpqa_group_col,
                    "unknown",
                ),
            seed=stable_seed(
                SEED,
                "gpqa",
            ),
        )
    else:
        gpqa_selected = stable_sample(
            gpqa_records,
            50,
            seed=stable_seed(
                SEED,
                "gpqa",
            ),
        )

    gpqa_tasks = []

    for i, x in enumerate(
        gpqa_selected,
        start=1,
    ):

        question = str(
            x[question_col]
        )

        correct = str(
            x[correct_col]
        )

        choices = [
            correct,
            *[
                str(
                    x[col]
                )
                for col
                in incorrect_cols
            ],
        ]

        rng = random.Random(
            stable_seed(
                SEED,
                "gpqa-choice-order",
                question,
            )
        )

        rng.shuffle(choices)

        labels = [
            "A", "B", "C", "D"
        ]

        correct_label = labels[
            choices.index(correct)
        ]

        option_text = "\n".join(
            f"{label}. {choice}"
            for label, choice
            in zip(
                labels,
                choices,
            )
        )

        gpqa_tasks.append({
            "task_id":
                f"GPQA-D-{i:03d}",

            "dataset":
                "GPQA-Diamond",

            "dataset_revision":
                PINNED_REVISIONS[
                    "GPQA-Diamond"
                ],

            "category":
                (
                    x.get(
                        gpqa_group_col,
                        "unknown",
                    )
                    if gpqa_group_col
                    else "unknown"
                ),

            "source_id":
                sha256_text(
                    question
                ),

            "prompt":
                (
                    f"{question}\n\n"
                    f"{option_text}\n\n"
                    "Answer with the single correct option letter."
                ),

            "reference_answer":
                correct_label,

            "scoring":
                "exact_choice",

            "metadata_json":
                canonical_json({
                    "source_subset":
                        "diamond",
                    "choice_order_hash":
                        sha256_text(
                            canonical_json(
                                choices
                            )
                        ),
                }),
        })

    # -------------------------
    # GSM8K
    # -------------------------

    gsm_ds = load_hf(
        "openai/gsm8k",
        "test",
        PINNED_REVISIONS["GSM8K"],
        config="main",
    )

    assert len(gsm_ds) == 1319

    gsm_selected = stable_sample(
        [dict(x) for x in gsm_ds],
        75,
        seed=stable_seed(
            SEED,
            "gsm8k",
        ),
    )

    gsm_tasks = []

    for i, x in enumerate(
        gsm_selected,
        start=1,
    ):

        question = str(
            x["question"]
        )

        answer = (
            str(x["answer"])
            .split("####")[-1]
            .strip()
        )

        gsm_tasks.append({
            "task_id":
                f"GSM8K-{i:03d}",

            "dataset":
                "GSM8K",

            "dataset_revision":
                PINNED_REVISIONS[
                    "GSM8K"
                ],

            "category":
                "math_word_problem",

            "source_id":
                sha256_text(
                    question
                ),

            "prompt":
                question,

            "reference_answer":
                answer,

            "scoring":
                "gsm8k_numeric",

            "metadata_json":
                canonical_json({
                    "config": "main"
                }),
        })

    # -------------------------
    # IFEval
    # -------------------------

    ifeval_ds = load_hf(
        "google/IFEval",
        "train",
        PINNED_REVISIONS["IFEval"],
    )

    assert len(ifeval_ds) == 541

    ifeval_selected = stratified_sample(
        [dict(x) for x in ifeval_ds],
        75,
        key_fn=lambda x:
            len(
                x["instruction_id_list"]
            ),
        seed=stable_seed(
            SEED,
            "ifeval",
        ),
    )

    ifeval_tasks = []

    for x in ifeval_selected:

        ifeval_tasks.append({
            "task_id":
                f"IFEVAL-{int(x['key']):04d}",

            "dataset":
                "IFEval",

            "dataset_revision":
                PINNED_REVISIONS[
                    "IFEval"
                ],

            "category":
                "instruction_following",

            "source_id":
                str(x["key"]),

            "prompt":
                str(x["prompt"]),

            "reference_answer":
                None,

            "scoring":
                "ifeval_official",

            "metadata_json":
                canonical_json({
                    "key":
                        x["key"],
                    "instruction_id_list":
                        x["instruction_id_list"],
                    "kwargs":
                        x["kwargs"],
                }),
        })

    # -------------------------
    # LiveCodeBench v6 — STREAMING SOURCE BUILD
    # -------------------------

    # The current Hugging Face `datasets` package cannot execute the
    # LiveCodeBench loading script. We therefore fetch the pinned release
    # files directly. We DO NOT keep all 1,055 full problem records in RAM.
    # First pass: retain only lightweight metadata for deterministic sampling.
    # Second pass: reread only the selected JSONL records.
    lcb_files = [
        "test.jsonl", "test2.jsonl", "test3.jsonl",
        "test4.jsonl", "test5.jsonl", "test6.jsonl",
    ]

    lcb_index = []
    lcb_local_paths = {}

    for filename in lcb_files:
        lcb_path = hf_hub_download(
            repo_id=LCB_REPO,
            filename=filename,
            repo_type="dataset",
            revision=PINNED_REVISIONS["LiveCodeBench"],
            token=HF_TOKEN,
        )
        lcb_local_paths[filename] = Path(lcb_path)

        with open(lcb_path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                x = json.loads(line)
                lcb_index.append({
                    "filename": filename,
                    "line_no": line_no,
                    "platform": str(x.get("platform", "unknown")),
                    "difficulty": str(x.get("difficulty", "unknown")),
                })
                del x

        gc.collect()

    assert len(lcb_index) == 1055, f"Unexpected LiveCodeBench release_v6 size: {len(lcb_index)}"

    lcb_selected_index = stratified_sample(
        lcb_index,
        100,
        key_fn=lambda x: (x["platform"], x["difficulty"]),
        seed=stable_seed(SEED, "livecodebench-v6"),
    )

    wanted = defaultdict(set)
    for row in lcb_selected_index:
        wanted[row["filename"]].add(row["line_no"])

    lcb_selected = []
    for filename, line_numbers in wanted.items():
        with open(lcb_local_paths[filename], "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f):
                if line_no not in line_numbers:
                    continue
                line = line.strip()
                if line:
                    lcb_selected.append(json.loads(line))

    assert len(lcb_selected) == 100

    # Restore the same deterministic order produced by stratified_sample.
    selected_keys = {(x["filename"], x["line_no"]): i for i, x in enumerate(lcb_selected_index)}
    # Attach source location while rereading so order is reproducible.
    lcb_tasks = []
    selected_lookup = {}
    for row in lcb_selected_index:
        selected_lookup[(row["filename"], row["line_no"])] = row

    # Re-read selected records in source order, then order by the sampled index.
    selected_records = {}
    for filename, line_numbers in wanted.items():
        with open(lcb_local_paths[filename], "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f):
                if line_no in line_numbers:
                    selected_records[(filename, line_no)] = json.loads(line)

    for i, idx_row in enumerate(lcb_selected_index, start=1):
        x = selected_records[(idx_row["filename"], idx_row["line_no"])]
        question_id = str(x.get("question_id") or x.get("id") or i)
        title = str(x.get("question_title", ""))
        content = str(x.get("question_content", ""))
        starter = str(x.get("starter_code", ""))
        prompt = f"{title}\n\n{content}"
        if starter.strip():
            prompt += "\n\nStarter code:\n" + starter
        lcb_tasks.append({
            "task_id": f"LCB-V6-{i:03d}",
            "dataset": "LiveCodeBench",
            "dataset_revision": PINNED_REVISIONS["LiveCodeBench"],
            "category": "code_generation",
            "source_id": question_id,
            "prompt": prompt,
            "reference_answer": None,
            "scoring": "livecodebench_official",
            "metadata_json": canonical_json({
                "question_id": question_id,
                "question_title": title,
                "platform": x.get("platform"),
                "difficulty": x.get("difficulty"),
                "starter_code": starter,
                "public_test_cases": x.get("public_test_cases"),
                "metadata": x.get("metadata"),
                "release": LCB_RELEASE,
            }),
        })

    del lcb_selected, selected_records, lcb_index, lcb_selected_index
    gc.collect()

    # -------------------------
    # BFCL
    # -------------------------

    # Download the exact BFCL commit archive instead of cloning/checking out.
    # This avoids the failure mode where an existing partial git repository
    # cannot checkout the requested commit.
    bfcl_dir = ROOT / "bfcl_repo"
    bfcl_archive = ROOT / f"bfcl_{BFCL_COMMIT}.tar.gz"
    bfcl_url = f"https://github.com/EnlightenedAI/BFCL/archive/{BFCL_COMMIT}.tar.gz"

    if not bfcl_dir.exists():
        if not bfcl_archive.exists():
            with requests.get(bfcl_url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with bfcl_archive.open("wb") as out:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            out.write(chunk)

        with tarfile.open(bfcl_archive, "r:gz") as tar:
            tar.extractall(ROOT)

        extracted = ROOT / f"BFCL-{BFCL_COMMIT}"
        if not extracted.exists():
            candidates = [p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("BFCL-")]
            if not candidates:
                raise RuntimeError("BFCL archive extracted but repository directory was not found.")
            extracted = candidates[0]
        extracted.rename(bfcl_dir)

    bfcl_data_dir = (
        bfcl_dir
        / "berkeley-function-call-leaderboard"
        / "bfcl_eval"
        / "data"
    )
    if not bfcl_data_dir.exists():
        # Some BFCL snapshots place the package at repository root.
        alternate = bfcl_dir / "bfcl_eval" / "data"
        if alternate.exists():
            bfcl_data_dir = alternate
        else:
            raise RuntimeError(f"BFCL data directory not found under {bfcl_dir}")

    bfcl_data_dir = (
        bfcl_dir
        / "berkeley-function-call-leaderboard"
        / "bfcl_eval"
        / "data"
    )

    bfcl_targets = {
        "simple_python": 20,
        "multiple": 10,
        "parallel": 10,
        "parallel_multiple": 10,
    }

    bfcl_tasks = []

    def read_jsonl(path):
        rows = []
        with path.open(
            "r",
            encoding="utf-8",
        ) as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(
                        json.loads(line)
                    )
        return rows

    bfcl_type_remap = {
        "dict": "object",
        "float": "number",
        "tuple": "array",
        "any": "string",
    }

    for category, target in (
        bfcl_targets.items()
    ):

        data_path = (
            bfcl_data_dir
            / f"BFCL_v4_{category}.json"
        )

        answer_path = (
            bfcl_data_dir
            / "possible_answer"
            / f"BFCL_v4_{category}.json"
        )

        assert data_path.exists()
        assert answer_path.exists()

        data_rows = read_jsonl(
            data_path
        )

        answer_rows = read_jsonl(
            answer_path
        )

        answers_by_id = {
            str(x["id"]): x
            for x in answer_rows
        }

        selected = stable_sample(
            data_rows,
            target,
            seed=stable_seed(
                SEED,
                "bfcl-v4",
                category,
            ),
        )

        for item in selected:

            source_id = str(
                item["id"]
            )

            assert (
                source_id
                in answers_by_id
            )

            bfcl_tasks.append({
                "task_id":
                    f"BFCL-{category.upper()}-{source_id}",

                "dataset":
                    "BFCL",

                "dataset_revision":
                    BFCL_COMMIT,

                "category":
                    category,

                "source_id":
                    source_id,

                "prompt":
                    canonical_json(
                        item["question"]
                    ),

                "reference_answer":
                    None,

                "scoring":
                    "bfcl_official",

                "metadata_json":
                    canonical_json({
                        "bfcl_id":
                            source_id,
                        "question":
                            item["question"],
                        "function":
                            item["function"],
                        "ground_truth":
                            answers_by_id[
                                source_id
                            ],
                        "category":
                            category,
                    }),
            })

    assert len(bfcl_tasks) == 50

    # -------------------------
    # FRES v1
    # -------------------------

    FRES_B64 = r'''
...
'''

fres_bytes = base64.b64decode(FRES_B64)

fres_rows = [
    json.loads(line)
    for line in fres_bytes.decode(
        "utf-8"
    ).splitlines()
    if line.strip()
]

assert len(fres_rows) == 50

fres_tasks = []

for x in fres_rows:

    fres_tasks.append({
        "task_id":
            x["task_id"],

        "dataset":
            "FRES",

        "dataset_revision":
            "v1",

        "category":
            x["group"],

        "source_id":
            x["task_id"],

        "prompt":
            x["prompt"],

        "reference_answer":
            x["expected_output"],

        "scoring":
            "fres_rubric",

        "metadata_json":
            canonical_json({
                "group":
                    x["group"],
                "scoring_rubric":
                    x["scoring_rubric"],
                "primary_capability":
                    x.get(
                        "primary_capability"
                    ),
            }),
    })

    # end task loop

    # no-op

assert len(fres_tasks) == 50

all_tasks = (
    mmlu_tasks
    + gpqa_tasks
    + gsm_tasks
    + ifeval_tasks
    + lcb_tasks
    + bfcl_tasks
    + fres_tasks
)

assert len(all_tasks) == 500

for task in all_tasks:
    task["prompt_hash"] = sha256_text(
        task["prompt"]
    )

all_tasks = sorted(
    all_tasks,
    key=lambda x: (
        x["dataset"],
        x["task_id"],
    ),
)

for index, task in enumerate(
    all_tasks,
    start=1,
):
    task["benchmark_index"] = index

assert len({
    x["task_id"]
    for x in all_tasks
}) == 500

# Write benchmark files.
with BENCHMARK_JSONL.open(
    "w",
    encoding="utf-8",
) as f:

    for task in all_tasks:

        f.write(
            json.dumps(
                task,
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )

fields = [
    "benchmark_index",
    "task_id",
    "dataset",
    "dataset_revision",
    "category",
    "source_id",
    "prompt",
    "reference_answer",
    "scoring",
    "prompt_hash",
    "metadata_json",
]

with BENCHMARK_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fields,
    )

    writer.writeheader()

    for task in all_tasks:
        writer.writerow({
            field:
                task.get(field)
            for field in fields
        })

public_df = pd.DataFrame(
    all_tasks
)[[
    "benchmark_index",
    "task_id",
    "dataset",
    "dataset_revision",
    "category",
    "source_id",
    "prompt_hash",
    "scoring",
]]

public_df.to_csv(
    PUBLIC_MANIFEST_CSV,
    index=False,
)

benchmark_sha = sha256_bytes(
    BENCHMARK_JSONL.read_bytes()
)

fres_sha = sha256_bytes(
    fres_bytes
)

frozen_meta = {
    "experiment_id":
        EXPERIMENT_ID,

    "freeze_date":
        "2026-09-26",

    "selection_seed":
        SEED,

    "total_tasks":
        500,

    "counts":
        dict(
            Counter(
                x["dataset"]
                for x in all_tasks
            )
        ),

    "benchmark_sha256":
        benchmark_sha,

    "fres_sha256":
        fres_sha,

    "sources": {
        "MMLU-Pro": {
            "repo":
                "TIGER-Lab/MMLU-Pro",
            "revision":
                PINNED_REVISIONS[
                    "MMLU-Pro"
                ],
            "split":
                "test",
            "selection_seed":
                stable_seed(
                    SEED,
                    "mmlu_pro",
                ),
            "count":
                100,
        },

        "GPQA-Diamond": {
            "repo":
                "Idavidrein/gpqa",
            "revision":
                PINNED_REVISIONS[
                    "GPQA-Diamond"
                ],
            "subset":
                "diamond",
            "selection_seed":
                stable_seed(
                    SEED,
                    "gpqa",
                ),
            "count":
                50,
            "redistribution":
                "restricted",
        },

        "GSM8K": {
            "repo":
                "openai/gsm8k",
            "revision":
                PINNED_REVISIONS[
                    "GSM8K"
                ],
            "config":
                "main",
            "split":
                "test",
            "selection_seed":
                stable_seed(
                    SEED,
                    "gsm8k",
                ),
            "count":
                75,
        },

        "IFEval": {
            "repo":
                "google/IFEval",
            "revision":
                PINNED_REVISIONS[
                    "IFEval"
                ],
            "split":
                "train",
            "selection_seed":
                stable_seed(
                    SEED,
                    "ifeval",
                ),
            "count":
                75,
        },

        "LiveCodeBench": {
            "repo":
                LCB_REPO,
            "revision":
                PINNED_REVISIONS[
                    "LiveCodeBench"
                ],
            "release":
                LCB_RELEASE,
            "selection_seed":
                stable_seed(
                    SEED,
                    "livecodebench-v6",
                ),
            "count":
                100,
        },

        "BFCL": {
            "repo":
                BFCL_REPO_URL,
            "commit":
                BFCL_COMMIT,
            "categories":
                bfcl_targets,
            "count":
                50,
        },

        "FRES": {
            "version":
                "v1",
            "sha256":
                fres_sha,
            "count":
                50,
        },
    },

    "status":
        "FROZEN",
}

write_json(
    FROZEN_MANIFEST,
    frozen_meta,
)

benchmark_df = pd.DataFrame(
    all_tasks
)

print(
    "FROZEN BENCHMARK CREATED"
)
print(
    "Tasks:",
    len(benchmark_df)
)
print(
    benchmark_df[
        "dataset"
    ].value_counts()
)
print(
    "SHA-256:",
    benchmark_sha,
)

# Release any large assembly objects before production.
for _name in [
    "mmlu_ds", "mmlu_rows", "mmlu_selected", "mmlu_tasks",
    "gpqa_df", "gpqa_records", "gpqa_selected", "gpqa_tasks",
    "gsm_ds", "gsm_selected", "gsm_tasks",
    "ifeval_ds", "ifeval_selected", "ifeval_tasks",
    "lcb_ds", "lcb_rows", "lcb_selected", "lcb_tasks",
    "bfcl_tasks", "data_rows", "answer_rows", "answers_by_id",
    "fres_rows", "fres_tasks", "all_tasks", "public_df",
]:
    globals().pop(_name, None)

gc.collect()

print("Assembly/recovery memory cleanup complete.")




No frozen benchmark found. Building it once from pinned sources.


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.14M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/42.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/12032 [00:00<?, ? examples/s]

...

test4.jsonl:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

test5.jsonl:   0%|          | 0.00/558M [00:00<?, ?B/s]

test6.jsonl:   0%|          | 0.00/134M [00:00<?, ?B/s]

/tmp/ipykernel_1731/3264574792.py:672: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(ROOT)


FROZEN BENCHMARK CREATED
Tasks: 500
dataset
MMLU-Pro         100
LiveCodeBench    100
IFEval            75
GSM8K             75
GPQA-Diamond      50
BFCL              50
FRES              50
Name: count, dtype: int64
SHA-256: bf904af6e4a7f909c92642f2299f1001c7f1aff88894684fd61f2e799df72bfb
Assembly/recovery memory cleanup complete.


In [ ]:
# ============================================================
# 5. STREAMING BENCHMARK INTEGRITY GATE
# ============================================================

if not BENCHMARK_JSONL.exists() or not FROZEN_MANIFEST.exists():
    raise RuntimeError("Benchmark bootstrap did not create the frozen benchmark files.")

frozen_meta = read_json(FROZEN_MANIFEST)
actual_sha = sha256_file_streaming(BENCHMARK_JSONL)
expected_sha = frozen_meta.get("benchmark_sha256")

if expected_sha and actual_sha != expected_sha:
    raise RuntimeError("FROZEN BENCHMARK SHA-256 MISMATCH. Refusing to run.")

task_ids = set()
dataset_counts = Counter()

with BENCHMARK_JSONL.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        task = json.loads(line)
        task_id = task.get("task_id")
        dataset = task.get("dataset")
        if not task_id or not dataset:
            raise RuntimeError(f"Malformed benchmark row at line {line_no}")
        if task_id in task_ids:
            raise RuntimeError(f"Duplicate task ID: {task_id}")
        if not task.get("prompt") and dataset != "BFCL":
            raise RuntimeError(f"Blank prompt at {task_id}")
        task_ids.add(task_id)
        dataset_counts[dataset] += 1

assert len(task_ids) == EXPECTED_TASKS
for dataset, expected in COUNTS.items():
    assert dataset_counts[dataset] == expected, (dataset, dataset_counts[dataset], expected)

print("FROZEN BENCHMARK VERIFIED")
print("Tasks:", len(task_ids))
print("SHA-256:", actual_sha)
for dataset in COUNTS:
    print(f"  {dataset:18s} {dataset_counts[dataset]:3d}")




FROZEN BENCHMARK VERIFIED
Tasks: 500
SHA-256: bf904af6e4a7f909c92642f2299f1001c7f1aff88894684fd61f2e799df72bfb
  MMLU-Pro           100
  GPQA-Diamond        50
  GSM8K               75
  IFEval              75
  LiveCodeBench      100
  BFCL                50
  FRES                50


In [ ]:
# ============================================================
# 7. FIKRA MODEL REGISTRY
# ============================================================

HEADERS = {
    "Authorization":
        f"Bearer {FIKRA_API_KEY}",
    "Content-Type":
        "application/json",
}

SESSION = requests.Session()
SESSION.headers.update(
    HEADERS
)

registry_response = SESSION.get(
    f"{FIKRA_BASE_URL}/models",
    timeout=30,
)

registry_response.raise_for_status()

registry_payload = (
    registry_response.json()
)

write_json(
    MODEL_REGISTRY_SNAPSHOT,
    registry_payload,
)

def extract_model_ids(payload):

    ids = []

    for item in (
        payload.get("data", [])
        if isinstance(payload, dict)
        else []
    ):

        if isinstance(item, dict):
            model_id = item.get("id")

            if model_id:
                ids.append(
                    model_id
                )

    return sorted(
        set(ids)
    )

LIVE_MODEL_IDS = extract_model_ids(
    registry_payload
)

missing = [
    model
    for model in FINAL_MODELS
    if model not in LIVE_MODEL_IDS
]

if missing:
    raise RuntimeError(
        "Required benchmark models are missing "
        "from the live Fikra registry: "
        + ", ".join(missing)
    )

print("Four-model registry gate: PASS")
for model in FINAL_MODELS:
    print(
        f"  {model:20s} → "
        f"{MODEL_UPSTREAM[model]}"
    )




Four-model registry gate: PASS
  fikra-flash          → GPT-OSS 20B
  fikra-pro-20b        → Qwen3.8 27B
  fikra-pro-120b       → GPT-OSS 120B
  fikra-qwen-max       → Qwen 3.8 Max


In [ ]:
# ============================================================
# 8. OPENAI-COMPATIBLE / BFCL REQUEST ADAPTER
# ============================================================

BFCL_TYPE_REMAP = {
    "dict": "object",
    "float": "number",
    "tuple": "array",
    "any": "string",
}

def normalize_bfcl_schema(node):

    if isinstance(node, dict):

        output = {}

        for key, value in node.items():

            if (
                key == "type"
                and isinstance(
                    value,
                    str,
                )
            ):
                output[key] = (
                    BFCL_TYPE_REMAP.get(
                        value,
                        value,
                    )
                )

            else:
                output[key] = (
                    normalize_bfcl_schema(
                        value
                    )
                )

        return output

    if isinstance(node, list):

        return [
            normalize_bfcl_schema(x)
            for x in node
        ]

    return node

def normalize_bfcl_tools(
    functions
):
    return [
        {
            "type": "function",
            "function":
                normalize_bfcl_schema(
                    fn
                ),
        }
        for fn in functions
    ]

def normalize_bfcl_messages(
    question
):

    if isinstance(question, list):

        if all(
            isinstance(x, list)
            for x in question
        ):
            flattened = []

            for conversation in question:
                flattened.extend(
                    conversation
                )

            question = flattened

        messages = []

        for turn in question:

            if not isinstance(
                turn,
                dict,
            ):
                raise ValueError(
                    f"Unexpected BFCL turn: {turn}"
                )

            messages.append({
                "role":
                    turn.get(
                        "role",
                        "user",
                    ),

                "content":
                    turn.get(
                        "content",
                        "",
                    ),
            })

        return messages

    return [{
        "role":
            "user",

        "content":
            str(question),
    }]

def build_request_payload(
    task,
    model,
):

    if task["dataset"] == "BFCL":

        meta = json.loads(
            task["metadata_json"]
        )

        return {
            "model":
                model,

            "messages":
                normalize_bfcl_messages(
                    meta["question"]
                ),

            "tools":
                normalize_bfcl_tools(
                    meta["function"]
                ),

            "tool_choice":
                "auto",

            "stream":
                True,

            "stream_options": {
                "include_usage":
                    True
            },

            "temperature":
                0,

            "top_p":
                1,

            "max_tokens":
                MAX_OUTPUT_TOKENS,
        }

    return {
        "model":
            model,

        "messages": [{
            "role":
                "user",
            "content":
                task["prompt"],
        }],

        "stream":
            True,

        "stream_options": {
            "include_usage":
                True
        },

        "temperature":
            0,

        "top_p":
            1,

        "max_tokens":
            MAX_OUTPUT_TOKENS,
    }

print("Request adapter ready.")




Request adapter ready.


In [ ]:
# ============================================================
# 9. STREAM PARSER + SINGLE-EVALUATION RUNNER
# ============================================================

RETRYABLE_STATUS = {
    408,
    429,
    500,
    502,
    503,
    504,
}

def parse_stream_event(
    event,
    content_parts,
    tool_calls,
):

    finish_reason = None

    for choice in (
        event.get("choices")
        or []
    ):

        delta = (
            choice.get("delta")
            or {}
        )

        content = delta.get(
            "content"
        )

        if content:
            content_parts.append(
                content
            )

        for tc in (
            delta.get(
                "tool_calls"
            )
            or []
        ):

            index = tc.get(
                "index",
                0,
            )

            if index not in tool_calls:
                tool_calls[index] = {
                    "id": "",
                    "type": "function",
                    "function": {
                        "name": "",
                        "arguments": "",
                    },
                }

            if tc.get("id"):
                tool_calls[index]["id"] = (
                    tc["id"]
                )

            if tc.get("type"):
                tool_calls[index]["type"] = (
                    tc["type"]
                )

            fd = (
                tc.get("function")
                or {}
            )

            if fd.get("name"):
                tool_calls[index][
                    "function"
                ]["name"] += fd["name"]

            if fd.get("arguments"):
                tool_calls[index][
                    "function"
                ]["arguments"] += (
                    fd["arguments"]
                )

        if choice.get(
            "finish_reason"
        ):
            finish_reason = choice[
                "finish_reason"
            ]

    return finish_reason

def run_fikra_task(
    task,
    model,
    max_attempts=4,
):

    payload = build_request_payload(
        task,
        model,
    )

    for attempt in range(
        1,
        max_attempts + 1,
    ):

        started_wall = (
            datetime.now(
                timezone.utc
            )
        )

        start_perf = (
            time.perf_counter()
        )

        first_token_perf = None

        output_parts = []
        tool_calls = {}

        usage = None
        finish_reason = None

        http_status = None
        error_type = None
        error_message = None

        response_headers = {}

        try:

            with SESSION.post(
                f"{FIKRA_BASE_URL}/chat/completions",
                json=payload,
                stream=True,
                timeout=(
                    30,
                    600,
                ),
            ) as response:

                http_status = (
                    response.status_code
                )

                response_headers = {
                    k: v
                    for k, v
                    in response.headers.items()
                    if (
                        k.lower().startswith(
                            "x-ratelimit-"
                        )
                        or
                        k.lower()
                        in {
                            "request-id",
                            "x-request-id",
                        }
                    )
                }

                if http_status != 200:

                    error_text = (
                        response.text[
                            :5000
                        ]
                    )

                    result = {
                        "success":
                            False,

                        "task_id":
                            task["task_id"],

                        "model":
                            model,

                        "attempt":
                            attempt,

                        "http_status":
                            http_status,

                        "error_type":
                            "http_error",

                        "error_message":
                            error_text,

                        "started_at":
                            started_wall.isoformat(),

                        "completed_at":
                            datetime.now(
                                timezone.utc
                            ).isoformat(),

                        "ttft_ms":
                            None,

                        "total_latency_ms":
                            (
                                time.perf_counter()
                                - start_perf
                            ) * 1000,

                        "output_text":
                            "",

                        "tool_calls_json":
                            None,

                        "input_tokens":
                            None,

                        "output_tokens":
                            None,

                        "reasoning_tokens":
                            None,

                        "queue_time_s": None,

                        "prompt_time_s": None,

                        "completion_time_s": None,

                        "api_total_time_s": None,

                        "total_tokens":
                            None,

                        "e2e_tokens_per_second":
                            None,

                        "api_completion_tokens_per_second":
                            None,

                        "finish_reason":
                            None,

                        "usage_json":
                            None,

                        "rate_limit_headers_json":
                            json.dumps(
                                response_headers
                            ),

                        "attempts_used":
                            attempt,
                    }

                    if (
                        http_status
                        in RETRYABLE_STATUS
                        and attempt
                        < max_attempts
                    ):

                        time.sleep(
                            min(
                                30,
                                2 ** (
                                    attempt - 1
                                ),
                            )
                            + random.uniform(
                                0.25,
                                1.0,
                            )
                        )

                        continue

                    return result

                for raw_line in (
                    response.iter_lines(
                        decode_unicode=True
                    )
                ):

                    if not raw_line:
                        continue

                    line = raw_line.strip()

                    if line.startswith(
                        "data:"
                    ):
                        line = line[
                            5:
                        ].strip()

                    if line == "[DONE]":
                        break

                    try:
                        event = json.loads(
                            line
                        )
                    except json.JSONDecodeError:
                        continue

                    if event.get(
                        "usage"
                    ) is not None:
                        usage = event[
                            "usage"
                        ]

                    before_content = (
                        len(output_parts)
                    )

                    before_tools = (
                        len(tool_calls)
                    )

                    finish_from_event = (
                        parse_stream_event(
                            event,
                            output_parts,
                            tool_calls,
                        )
                    )

                    if finish_from_event:
                        finish_reason = (
                            finish_from_event
                        )

                    if (
                        first_token_perf
                        is None
                        and (
                            len(output_parts)
                            > before_content
                            or
                            len(tool_calls)
                            > before_tools
                        )
                    ):
                        first_token_perf = (
                            time.perf_counter()
                        )

            total_latency_ms = (
                time.perf_counter()
                - start_perf
            ) * 1000

            ttft_ms = (
                (
                    first_token_perf
                    - start_perf
                ) * 1000
                if first_token_perf
                is not None
                else None
            )

            output_text = "".join(
                output_parts
            )

            normalized_tool_calls = list(
                tool_calls.values()
            )

            input_tokens = None
            output_tokens = None
            reasoning_tokens = None
            total_tokens = None

            if usage:

                input_tokens = usage.get(
                    "prompt_tokens"
                )

                output_tokens = usage.get(
                    "completion_tokens"
                )

                total_tokens = usage.get(
                    "total_tokens"
                )

                details = (
                    usage.get(
                        "completion_tokens_details"
                    )
                    or {}
                )

                reasoning_tokens = (
                    details.get(
                        "reasoning_tokens"
                    )
                )

                queue_time_s = usage.get("queue_time")
                prompt_time_s = usage.get("prompt_time")
                completion_time_s = usage.get("completion_time")
                api_total_time_s = usage.get("total_time")

            tokens_per_second = None
            api_completion_tokens_per_second = None

            if (
                output_tokens
                is not None
                and total_latency_ms > 0
            ):
                tokens_per_second = (
                    output_tokens
                    /
                    (
                        total_latency_ms
                        / 1000
                    )
                )

            if (
                output_tokens is not None
                and completion_time_s
                and completion_time_s > 0
            ):
                api_completion_tokens_per_second = (
                    output_tokens / completion_time_s
                )

            first_token_at = None
            if ttft_ms is not None:
                first_token_at = (
                    started_wall
                    + timedelta(milliseconds=ttft_ms)
                ).isoformat()

            return {
                "success":
                    True,

                "task_id":
                    task["task_id"],

                "model":
                    model,

                "attempt":
                    attempt,

                "http_status":
                    http_status,

                "error_type":
                    None,

                "error_message":
                    None,

                "started_at":
                    started_wall.isoformat(),

                "first_token_at":
                    first_token_at,

                "completed_at":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),

                "ttft_ms":
                    ttft_ms,

                "total_latency_ms":
                    total_latency_ms,

                "output_text":
                    output_text,

                "tool_calls_json":
                    (
                        json.dumps(
                            normalized_tool_calls,
                            ensure_ascii=False,
                        )
                        if normalized_tool_calls
                        else None
                    ),

                "input_tokens":
                    input_tokens,

                "output_tokens":
                    output_tokens,

                "reasoning_tokens":
                    reasoning_tokens,

                "queue_time_s":
                    queue_time_s,

                "prompt_time_s":
                    prompt_time_s,

                "completion_time_s":
                    completion_time_s,

                "api_total_time_s":
                    api_total_time_s,

                "total_tokens":
                    total_tokens,

                "e2e_tokens_per_second":
                    tokens_per_second,

                "api_completion_tokens_per_second":
                    api_completion_tokens_per_second,

                "finish_reason":
                    finish_reason,

                "usage_json":
                    (
                        json.dumps(
                            usage,
                            ensure_ascii=False,
                        )
                        if usage is not None
                        else None
                    ),

                "rate_limit_headers_json":
                    json.dumps(
                        response_headers
                    ),

                "attempts_used":
                    attempt,
            }

        except requests.exceptions.Timeout as e:

            error_type = "timeout"
            error_message = str(e)

        except requests.exceptions.RequestException as e:

            error_type = "request_exception"
            error_message = str(e)

        except Exception as e:

            error_type = type(e).__name__
            error_message = str(e)

        if attempt < max_attempts:

            time.sleep(
                min(
                    30,
                    2 ** (
                        attempt - 1
                    ),
                )
                + random.uniform(
                    0.25,
                    1.0,
                )
            )

        else:

            return {
                "success":
                    False,

                "task_id":
                    task["task_id"],

                "model":
                    model,

                "attempt":
                    attempt,

                "http_status":
                    http_status,

                "error_type":
                    error_type,

                "error_message":
                    error_message,

                "started_at":
                    started_wall.isoformat(),

                "first_token_at":
                    None,

                "completed_at":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),

                "ttft_ms":
                    None,

                "total_latency_ms":
                    (
                        time.perf_counter()
                        - start_perf
                    ) * 1000,

                "output_text":
                    "",

                "tool_calls_json":
                    None,

                "input_tokens":
                    None,

                "output_tokens":
                    None,

                "reasoning_tokens":
                    None,

                "queue_time_s": None,

                "prompt_time_s": None,

                "completion_time_s": None,

                "api_total_time_s": None,

                "total_tokens":
                    None,

                "e2e_tokens_per_second":
                    None,

                "finish_reason":
                    None,

                "usage_json":
                    None,

                "rate_limit_headers_json":
                    json.dumps(
                        response_headers
                    ),

                "attempts_used":
                    attempt,
            }

if "run_fikra_task" not in globals():
    raise RuntimeError("Runner definition failed.")

print("Runner ready.")




Runner ready.


In [ ]:
# ============================================================
# 8. ZERO-COST PRE-FLIGHT
# ============================================================

# Verify the final four model IDs again. No benchmark calls are made here.
missing = [m for m in FINAL_MODELS if m not in LIVE_MODEL_IDS]
if missing:
    raise RuntimeError("Required model IDs missing from the live registry: " + ", ".join(missing))

# Persist the fact that the production runner is using the already-validated adapter.
VALIDATION_FLAG = ROOT / "validation_complete.json"
write_json(VALIDATION_FLAG, {
    "validated_before_production": True,
    "validation_type": "BFCL smoke test + 20-call cross-benchmark pilot",
    "models": FINAL_MODELS,
    "timestamp": datetime.now(timezone.utc).isoformat(),
})

print("Zero-cost pre-flight PASSED.")
print("No API validation calls made in this notebook.")




Zero-cost pre-flight PASSED.
No API validation calls made in this notebook.


In [ ]:
# ============================================================
# 11. PRODUCTION RUN MANIFEST
# ============================================================

if RUN_MANIFEST_PATH.exists():

    run_manifest = read_json(
        RUN_MANIFEST_PATH
    )

    assert (
        run_manifest["run_id"]
        == RUN_ID
    )

else:

    frozen_meta = read_json(
        FROZEN_MANIFEST
    )

    run_manifest = {
        "run_id":
            RUN_ID,

        "experiment_id":
            EXPERIMENT_ID,

        "benchmark_tasks":
            500,

        "models":
            FINAL_MODELS,

        "model_upstream":
            MODEL_UPSTREAM,

        "expected_evaluations":
            2000,

        "generation":
            {
                "temperature":
                    0,

                "top_p":
                    1,

                "max_tokens":
                    MAX_OUTPUT_TOKENS,

                "stream":
                    True,
            },

        "benchmark_sha256":
            frozen_meta[
                "benchmark_sha256"
            ],

        "bfcl_commit":
            BFCL_COMMIT,

        "started_at":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "status":
            "RUNNING",
    }

    write_json(
        RUN_MANIFEST_PATH,
        run_manifest,
    )

print(
    "Run ID:",
    RUN_ID,
)

print(
    "Results:",
    PRODUCTION_PATH,
)




Run ID: fikra-benchmark-phase1-2026-09-26-4model
Results: /content/fikra_benchmark_phase1/fikra-benchmark-phase1-2026-09-26-4model_results.jsonl


In [ ]:
# ============================================================
# 10. PARALLEL RESUMABLE 2,000-EVALUATION PRODUCTION RUN
# ============================================================
# Runs many requests at once (thread pool) under one shared rate limiter.
# Only the main thread writes to the results file, so rows never interleave.
# Safe to interrupt and re-run: finished (task_id, model) pairs are skipped.

import threading
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from requests.adapters import HTTPAdapter

MAX_WORKERS = 12      # simultaneous requests in flight
TARGET_RPM = 80       # shared cap across all workers (your limit is 100)

SESSION.mount("https://", HTTPAdapter(pool_connections=MAX_WORKERS * 2, pool_maxsize=MAX_WORKERS * 2))


class RateLimiter:
    def __init__(self, rpm):
        self.interval = 60.0 / rpm
        self.lock = threading.Lock()
        self.next_time = time.monotonic()

    def wait(self):
        with self.lock:
            now = time.monotonic()
            slot = max(now, self.next_time)
            self.next_time = slot + self.interval
        delay = slot - now
        if delay > 0:
            time.sleep(delay)

    def back_off(self):
        # Called when the API answers 429: slow the shared pace by 25% (max 1 req / 5 s).
        with self.lock:
            self.interval = min(self.interval * 1.25, 5.0)


limiter = RateLimiter(TARGET_RPM)

completed_keys = set()
existing_rows = 0
partial_lines = 0

if PRODUCTION_PATH.exists():
    with PRODUCTION_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            existing_rows += 1
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                partial_lines += 1
                continue
            if row.get("success"):
                completed_keys.add((row.get("task_id"), row.get("model")))

print("Existing result rows:", existing_rows)
print("Successful completed pairs:", len(completed_keys))
print("Remaining evaluations:", EXPECTED_EVALUATIONS - len(completed_keys))

if len(completed_keys) > EXPECTED_EVALUATIONS:
    raise RuntimeError("More successful evaluations exist than the 2,000-evaluation experiment allows.")


def pending_pairs():
    """Yield (task, model) for every evaluation not yet successful, one task in RAM at a time."""
    with BENCHMARK_JSONL.open("r", encoding="utf-8") as benchmark_file:
        for line in benchmark_file:
            if not line.strip():
                continue
            task = json.loads(line)
            task_id = task["task_id"]
            model_order = FINAL_MODELS.copy()
            random.Random(stable_seed(SEED, "production-model-order", task_id)).shuffle(model_order)
            for model in model_order:
                if (task_id, model) not in completed_keys:
                    yield task, model


def evaluate_one(task, model):
    limiter.wait()
    result = run_fikra_task(task, model, max_attempts=4)
    result.update({
        "run_id": RUN_ID,
        "benchmark_index": task.get("benchmark_index"),
        "dataset": task.get("dataset"),
        "category": task.get("category"),
        "source_id": task.get("source_id"),
        "prompt_hash": task.get("prompt_hash"),
        "scoring": task.get("scoring"),
        "dataset_revision": task.get("dataset_revision"),
    })
    return result


if len(completed_keys) == EXPECTED_EVALUATIONS:
    print("All 2,000 successful evaluations already exist. Nothing to run.")
else:
    processed = 0
    session_successes = 0
    session_failures = 0
    t_start = time.time()
    pairs = pending_pairs()
    exhausted = False
    inflight = {}

    executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    try:
        with PRODUCTION_PATH.open("a", encoding="utf-8") as output_file:
            while True:
                # Keep a small bounded queue so RAM stays low.
                while not exhausted and len(inflight) < MAX_WORKERS * 2:
                    try:
                        task, model = next(pairs)
                    except StopIteration:
                        exhausted = True
                        break
                    inflight[executor.submit(evaluate_one, task, model)] = (task["task_id"], model)

                if not inflight:
                    break

                done, _ = wait(list(inflight), return_when=FIRST_COMPLETED)

                for fut in done:
                    key = inflight.pop(fut)
                    try:
                        result = fut.result()
                    except Exception as e:
                        session_failures += 1
                        processed += 1
                        print(f"Worker error for {key}: {type(e).__name__}: {e}")
                        continue

                    output_file.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")
                    output_file.flush()
                    processed += 1

                    if result.get("success"):
                        completed_keys.add(key)
                        session_successes += 1
                    else:
                        session_failures += 1
                        if result.get("http_status") == 429:
                            limiter.back_off()

                    del result

                    if processed % 10 == 0:
                        os.fsync(output_file.fileno())

                    if processed % 25 == 0:
                        gc.collect()
                        elapsed = time.time() - t_start
                        rpm = processed / elapsed * 60 if elapsed > 0 else 0
                        remaining = EXPECTED_EVALUATIONS - len(completed_keys)
                        eta_min = remaining / rpm if rpm > 0 else float("inf")
                        write_json(PROGRESS_PATH, {
                            "run_id": RUN_ID,
                            "timestamp": datetime.now(timezone.utc).isoformat(),
                            "successful_total": len(completed_keys),
                            "session_successes": session_successes,
                            "session_failures": session_failures,
                            "remaining": remaining,
                        })
                        print(
                            f"CHECKPOINT | successful {len(completed_keys):4d}/2000 "
                            f"| remaining {remaining:4d} "
                            f"| session failures {session_failures:3d} "
                            f"| {rpm:5.1f} req/min | ETA ~{eta_min:5.1f} min"
                        )

            os.fsync(output_file.fileno())
    except KeyboardInterrupt:
        print("Interrupted. Progress is saved; re-run this cell to resume.")
        executor.shutdown(wait=False, cancel_futures=True)
        raise
    finally:
        executor.shutdown(wait=False, cancel_futures=True)

    print("Production pass ended.")
    print("Successful total:", len(completed_keys), "/ 2000")
    print("Remaining:", EXPECTED_EVALUATIONS - len(completed_keys))

Existing result rows: 0
Successful completed pairs: 0
Remaining evaluations: 2000
CHECKPOINT | successful   25/2000 | remaining 1975 | session failures   0 |  71.9 req/min | ETA ~ 27.5 min
CHECKPOINT | successful   50/2000 | remaining 1950 | session failures   0 |  76.5 req/min | ETA ~ 25.5 min
CHECKPOINT | successful   75/2000 | remaining 1925 | session failures   0 |  78.2 req/min | ETA ~ 24.6 min
CHECKPOINT | successful  100/2000 | remaining 1900 | session failures   0 |  76.0 req/min | ETA ~ 25.0 min
CHECKPOINT | successful  125/2000 | remaining 1875 | session failures   0 |  78.3 req/min | ETA ~ 24.0 min
CHECKPOINT | successful  149/2000 | remaining 1851 | session failures   1 |  79.5 req/min | ETA ~ 23.3 min
CHECKPOINT | successful  174/2000 | remaining 1826 | session failures   1 |  79.5 req/min | ETA ~ 23.0 min
CHECKPOINT | successful  199/2000 | remaining 1801 | session failures   1 |  79.1 req/min | ETA ~ 22.8 min
CHECKPOINT | successful  222/2000 | remaining 1778 | session f

In [ ]:
# ============================================================
# 11. STREAMING INTEGRITY CHECK + CSV EXPORT
# ============================================================

seen_successful = set()
total_rows = 0
successful_rows = 0
failed_rows = 0
partial_lines = 0
duplicate_successes = 0

with PRODUCTION_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            partial_lines += 1
            continue

        total_rows += 1
        key = (row.get("task_id"), row.get("model"))

        if row.get("success"):
            successful_rows += 1
            if key in seen_successful:
                duplicate_successes += 1
            seen_successful.add(key)
        else:
            failed_rows += 1

expected_successes = EXPECTED_EVALUATIONS
missing = expected_successes - len(seen_successful)

print("Raw result rows:", total_rows)
print("Successful rows:", successful_rows)
print("Failed rows:", failed_rows)
print("Partial final lines ignored:", partial_lines)
print("Duplicate successful pairs:", duplicate_successes)
print("Missing successful evaluations:", missing)

if duplicate_successes:
    raise RuntimeError("Duplicate successful task/model pairs detected.")

if missing == 0:
    print("RAW EXPERIMENT INTEGRITY: PASS")
else:
    print("Run remains resumable; rerun the notebook to continue.")

# Stream successful JSONL rows into CSV without pandas.
CSV_FIELDS = [
    "run_id", "task_id", "model", "benchmark_index", "dataset", "category",
    "source_id", "dataset_revision", "prompt_hash", "scoring", "success",
    "attempt", "http_status", "error_type", "error_message", "started_at",
    "completed_at", "ttft_ms", "total_latency_ms", "input_tokens",
    "output_tokens", "reasoning_tokens", "queue_time_s", "prompt_time_s",
    "completion_time_s", "api_total_time_s", "e2e_tokens_per_second",
    "api_completion_tokens_per_second", "finish_reason", "output_text",
    "tool_calls_json", "usage_json", "rate_limit_headers_json", "attempts_used",
]

with PRODUCTION_PATH.open("r", encoding="utf-8") as source, FINAL_CSV.open("w", newline="", encoding="utf-8") as target:
    writer = csv.DictWriter(target, fieldnames=CSV_FIELDS, extrasaction="ignore")
    writer.writeheader()
    for line in source:
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if row.get("success"):
            writer.writerow(row)

print("CSV:", FINAL_CSV)

if successful_rows == EXPECTED_EVALUATIONS:
    final_manifest = {
        "run_id": RUN_ID,
        "experiment_id": EXPERIMENT_ID,
        "status": "COMPLETE",
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "benchmark_tasks": EXPECTED_TASKS,
        "models": FINAL_MODELS,
        "successful_evaluations": successful_rows,
        "benchmark_sha256": actual_sha,
    }
    write_json(RUN_MANIFEST_PATH, final_manifest)
    print("\n========================================")
    print("FIKRA PHASE 1 RAW EXPERIMENT COMPLETE")
    print("2,000 / 2,000 successful evaluations")
    print("========================================")




Raw result rows: 2000
Successful rows: 1643
Failed rows: 357
Partial final lines ignored: 0
Duplicate successful pairs: 0
Missing successful evaluations: 357
Run remains resumable; rerun the notebook to continue.
CSV: /content/fikra_benchmark_phase1/fikra-benchmark-phase1-2026-09-26-4model_results.csv


In [ ]:
# ============================================================
# 12. LIGHTWEIGHT MODEL/DATASET SUMMARY (NO PANDAS)
# ============================================================

model_stats = defaultdict(lambda: {"n": 0, "ttft": [], "lat": [], "in": [], "out": [], "ok": 0})
dataset_counts = Counter()

with PRODUCTION_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if not row.get("success"):
            continue
        m = row["model"]
        d = row["dataset"]
        s = model_stats[m]
        s["n"] += 1
        s["ok"] += 1
        dataset_counts[d] += 1
        if row.get("ttft_ms") is not None: s["ttft"].append(row["ttft_ms"])
        if row.get("total_latency_ms") is not None: s["lat"].append(row["total_latency_ms"])
        if row.get("input_tokens") is not None: s["in"].append(row["input_tokens"])
        if row.get("output_tokens") is not None: s["out"].append(row["output_tokens"])

def median(values):
    values = sorted(values)
    if not values: return None
    n = len(values)
    mid = n // 2
    if n % 2: return values[mid]
    return (values[mid-1] + values[mid]) / 2

print("Dataset successful counts:")
for d in COUNTS:
    print(f"  {d:18s} {dataset_counts[d]:4d}")

print("\nModel execution medians:")
for m in FINAL_MODELS:
    s = model_stats[m]
    print(
        f"  {m:18s} n={s['n']:4d} "
        f"median_ttft={median(s['ttft'])!s:>10} ms "
        f"median_latency={median(s['lat'])!s:>10} ms "
        f"median_out={median(s['out'])!s:>8}"
    )

print("\nRaw experiment is ready for official scoring.")


Dataset successful counts:
  MMLU-Pro            266
  GPQA-Diamond        172
  GSM8K               258
  IFEval              274
  LiveCodeBench       286
  BFCL                199
  FRES                188

Model execution medians:
  fikra-flash        n= 444 median_ttft=1967.3989965000374 ms median_latency=3419.560017500089 ms median_out=   366.5
  fikra-pro-20b      n= 247 median_ttft=880.7190930001525 ms median_latency=1602.2192099999302 ms median_out=     139
  fikra-pro-120b     n= 452 median_ttft=2010.6853760000831 ms median_latency=3297.9107460000137 ms median_out=   410.0
  fikra-qwen-max     n= 500 median_ttft=13289.491240499956 ms median_latency=16065.409293499897 ms median_out=   529.5

Raw experiment is ready for official scoring.


In [ ]:
# ============================================================
# 13. ZIP + DOWNLOAD EVERYTHING
# ============================================================
import shutil
from google.colab import files

zip_path = "/content/fikra_benchmark_phase1.zip"
shutil.make_archive("/content/fikra_benchmark_phase1", "zip", ROOT)

print("Zipped:", zip_path)
files.download(zip_path)
